# Adversarial Attacks on Neural Networks

The goal of this notebook is to show different adversarial attacks.

The examples use the [CleverHans](https://github.com/cleverhans-lab/cleverhans) library to:

1. benchmark neural networks for vulnerability against specific attack types
2. crafting attacks

In [7]:
pip install -r requirements.txt


You should consider upgrading via the '/Users/jonasson/.pyenv/versions/datascience-in-prod/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [6]:
# Imports
import os
from easydict import EasyDict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import datasets
import matplotlib.pyplot as plt


from cleverhans.torch.attacks.fast_gradient_method import fast_gradient_method

/Users/jonasson/.pyenv/versions/datascience-in-prod/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/jonasson/.pyenv/versions/datascience-in-prod/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Data Loading

In [ ]:
def ld_mnist():
    train_transforms = torchvision.transforms.Compose([torchvision.transforms.ToTensor()])
    test_transforms = torchvision.transforms.Compose([torchvision.transforms.ToTensor()])

    # Load MNIST dataset
    train_dataset = torchvision.datasets.MNIST(root="/tmp/data", train = True, download = True, transform=train_transforms)
    test_dataset = torchvision.datasets.MNIST(root="/tmp/data", train=False, download = True, transform=test_transforms)

    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)
    
    return EasyDict(train=train_loader, test=test_loader)

In [ ]:
data = ld_mnist()
print("Data loaded.")

## The CNN Models

In [ ]:
# Very simple CNN for quick tests or if the other models are too intensive
class Very_Simple_CNN(torch.nn.Module):
    """Target CNN architecture."""
    def __init__(self, in_channels=1, p_drop=0.0): # 2 conv layers
        super(Basic_CNN, self).__init__()               # Kernel sizes
        self.conv1 = nn.Conv2d(in_channels, 64, 8, 1)   # (28x28) --> (21x21)
        self.conv2 = nn.Conv2d(64, 128, 6, 2)           # (21x21) --> (8x8)
        self.dropout = nn.Dropout(p=p_drop)
        self.fc1 = nn.Linear(128 * 4 * 4, 128)
        self.fc2 = nn.Linear(128, 10)
        
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc1(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# ---------------------------------

class Basic_CNN(torch.nn.Module):
    """Target CNN architecture."""

    def __init__(self, in_channels=1, p_drop=0.0): # 3 conv layers
        super(Basic_CNN, self).__init__()               # Kernel sizes
        self.conv1 = nn.Conv2d(in_channels, 64, 8, 1)   # (28x28) --> (21x21)
        self.conv2 = nn.Conv2d(64, 128, 6, 2)           # (21x21) --> (8x8)
        self.conv3 = nn.Conv2d(128, 128, 5, 1)          # (8x8) --> (4x4)
        self.dropout = nn.Dropout(p=p_drop)
        self.fc1 = nn.Linear(128 * 4 * 4, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc1(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x
    
# ---------------------------------

class Target_CNN(torch.nn.Module): # 4 conv layers
    """Basic CNN architecture."""

    def __init__(self, in_channels=1, p_drop=0.0):
        super(Target_CNN, self).__init__()              # Kernel sizes
        self.conv1 = nn.Conv2d(in_channels, 32, 5, 1)  # (28x28 -> 24x24)
        self.conv2 = nn.Conv2d(32, 64, 3, 2)           # (24x24 -> 11x11)
        self.conv3 = nn.Conv2d(64, 128, 3, 2)          # (11x11 -> 5x5)
        self.conv4 = nn.Conv2d(128, 128, 3, 1)         # (5x5 -> 3x3)
        self.dropout = nn.Dropout(p=p_drop)
        self.fc1 = nn.Linear(128 * 3 * 3, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc1(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [ ]:
import torch.optim as optim
from tqdm import tqdm

def train_model(model, train_loader, test_loader, epochs=5, lr=0.001, device="cpu"):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}: Loss = {running_loss/len(train_loader):.4f}")

        acc = evaluate_model(model, test_loader, device)
        print(f"Test Accuracy after epoch {epoch+1}: {acc:.2f}%")

    return model


def evaluate_model(model, test_loader, device="cpu"):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total


In [ ]:
# Only used while training the models for the first time
to_train = False

to_train = not (os.path.exists('basic_cnn.pth') or os.path.exists('target_cnn.pth'))

if to_train:
    print("\nTraining Basic_CNN...")
    basic_cnn = Basic_CNN()
    basic_cnn = train_model(basic_cnn, data.train, data.test, epochs=1)
    acc_basic = evaluate_model(basic_cnn, data.test)
    print(f"Final Test Accuracy (Basic_CNN): {acc_basic:.2f}%")
    torch.save(basic_cnn.state_dict(), "basic_cnn.pth")
    
    # Train Target CNN
    print("\nTraining Target_CNN...")
    target_cnn = Target_CNN()
    target_cnn = train_model(target_cnn, data.train, data.test, epochs=1)
    acc_target = evaluate_model(target_cnn, data.test)
    print(f"Final Test Accuracy (Target_CNN): {acc_target:.2f}%")
    torch.save(target_cnn.state_dict(), "target_cnn.pth")



You can load the pretrained models here using the template code below, they are located in the same directory as this exercise notebook.

In [ ]:
# Reload pretrained model
basic_cnn = Basic_CNN()
basic_cnn.load_state_dict(torch.load("basic_cnn.pth"))
basic_cnn.eval() # switch to evaluation mode

# Reload pretrained model
target_cnn = Target_CNN()
target_cnn.load_state_dict(torch.load("target_cnn.pth"))
target_cnn.eval() # switch to evaluation mode

# Adversarial Attack

We are ready to set up the experiment.

The parameters `eps` and `epsilon` are the values for adversarial perturbation. `0.01 - 0.1` are for subtle attacks. `0.3` is a high score and has been chosen to allow us to visually inspect the resulting images.

The FGSM attack happens after training and exploits the decision boundary of the model.

In [ ]:
def fgsm_attack(model, x, y, eps=0.3, norm=np.inf):
    """
    Run FGSM attack using CleverHans implementation.
    """
    device = x.device
    model.eval()

    adv = fast_gradient_method( # From cleverhans
        model_fn=model,
        x=x,
        eps=eps,
        norm=norm,
        y=y,
        clip_min=0.0,
        clip_max=1.0,
        targeted=False
    )

    delta = adv - x
    return adv.detach(), delta.detach()
# Future attacks to implement could be PGD, or CW also found in celeverhans


def visualize_attack(model, loader, eps=0.3, n_samples=3, title=""):
    device = "cpu"
    model.eval()
    x, y = next(iter(loader))
    x, y = x.to(device), y.to(device)
    adv, delta = fgsm_attack(model, x, y, eps=eps)

    for i in range(n_samples):
        orig = x[i].squeeze().cpu().numpy()
        pert = delta[i].squeeze().cpu().numpy()
        adv_img = adv[i].squeeze().cpu().numpy()

        pred_orig = model(x[i:i+1]).argmax(1).item()
        pred_adv = model(adv[i:i+1]).argmax(1).item()

        plt.figure(figsize=(9,3))
        plt.subplot(1,3,1)
        plt.imshow(orig, cmap="gray")
        plt.title(f"Original\npred: {pred_orig}")
        plt.axis("off")

        plt.subplot(1,3,2)
        plt.imshow(pert, cmap="seismic")
        plt.title("Perturbation")
        plt.axis("off")

        plt.subplot(1,3,3)
        plt.imshow(adv_img, cmap="gray")
        plt.title(f"Adversarial\npred: {pred_adv}")
        plt.axis("off")

        plt.suptitle(title + f" (eps={eps})")
        plt.show()

#### Visualizing the Adversarial Attacks

In the visualization we have:

1. The original MNIST image on the left.

2. The perturbation, which is noise that exploits the model's decision boundaeries.

3. The combined image between `1` and `2` on the right and the predicted value by the neural network.

As stated earlier we have chosen a high epsilon of `0.3` to allow us to visually inspect the result. In real-life attacks a much lower epsilon would be chosen to achieve the same result but with, to the human eye, imperceptible differences.

In [ ]:
print("Visualizing attacks on CNN...")
visualize_attack(basic_cnn, data.test, eps=0.3, n_samples=3)

### Questions

1. At what level of permutation (epsilon) does it become difficult for us humans to see it?

<!-- todo -->

# Poison Attack

Let's try poison attacks which occur during the training phase. 

We will specifically try a targeted attack. The goal is to make the model misidentify a label as another label. 

### Setting up the poison attacks

We will go with a `flip_rate=0.05`, meaning that `5%` of labels are flipped to other classes.

By experimenting with the `flip_rate`, we can see how few of the examples need to be "poisoned" in order to diminish the models accuracy.

We are trying `poison_random` where it chooses random samples to poison.

In [ ]:
from torch.utils.data import Dataset
import numpy as np

class PoisonedLabels(Dataset):
    def __init__(self, dataset, method="random", frac=0.1, seed=None, n_classes=10):
        """
        Wraps existing dataset and poisons labels using different strategies.

        Args:
            dataset: base dataset (e.g., torchvision.datasets.MNIST)
            method: "uniform", "random", or "gradient"
            frac: fraction of labels to flip
            seed: random seed for reproducibility
            n_classes: number of classes (default=10 for MNIST)
        """
        self.dataset = dataset
        self.method = method
        self.frac = frac
        self.n_classes = n_classes
        self.rng = np.random.default_rng(seed)

        # extract labels
        if hasattr(dataset, "targets"):
            y = np.array(dataset.targets)
        elif hasattr(dataset, "labels"):
            y = np.array(dataset.labels)

        # choose poisoning method
        if method == "uniform":
            y_poison, flip_idx = self._poison_uniform(y)
        elif method == "gradient":
            X = self._flatten_features(dataset)
            y_poison, flip_idx = self._poison_high_gradient(X, y)
        else:
            raise ValueError(f"Unknown poisoning method: {method}")

        self.targets = y_poison.tolist()
        self.flip_idx = flip_idx  # store indices that were flipped

    def _poison_uniform(self, y):
        """Flip a fraction of labels to a random *different* class."""
        n = len(y)
        n_flips = int(self.frac * n)
        flip_idx = self.rng.choice(n, size=n_flips, replace=False)

        y_flipped = y.copy()
        for i in flip_idx:
            new_label = self.rng.integers(0, self.n_classes)
            while new_label == y_flipped[i]:
                new_label = self.rng.integers(0, self.n_classes)
            y_flipped[i] = new_label
        return y_flipped, flip_idx

    def _flatten_features(self, dataset):
        """Convert dataset into flattened NumPy array for surrogate training."""
        X_list = []
        for x, _ in dataset:
            X_list.append(np.array(x).reshape(-1))
        return np.vstack(X_list)

    def __getitem__(self, idx):
        x, _ = self.dataset[idx]
        return x, self.targets[idx]

    def __len__(self):
        return len(self.dataset)


## Training of the Poison Models

The following code poisons the training data by flipping 5% of the labels to another class in order to misclassify them, then the model is retrained on the poisoned data.

We recommend to use the basic CNN model to retrain as it is slightly quicker in training than the other model.

In [ ]:
# testing clean model
base_acc_ = evaluate_model(basic_cnn, data.test)
print(f"Test Accuracy (Basic CNN): {base_acc_:.2f}%")

In [ ]:
# Apply poisoning
poisoned_dataset = PoisonedLabels(data.train.dataset, method="uniform", frac=0.5, seed=42)
poisoned_train_loader = torch.utils.data.DataLoader(poisoned_dataset, batch_size=128, shuffle=True)

# training poisoned model
model_to_poison = Basic_CNN()
poisoned_model = train_model(model_to_poison, poisoned_train_loader, data.test, epochs=1)
poisoned_acc_ = evaluate_model(poisoned_model, data.test)

In [ ]:
print(f"Test Accuracy (Basic CNN): {base_acc_:.2f}%")
print(f"Test Accuracy (Poisoned Model): {poisoned_acc_:.2f}%")

## Questions
How few % of the data have you found is enough to render the model essentially useless?

1. Can you think of reasons for why it fails at that low procent, model complexity, data size etc.
2. Do you see any different in the amount needed from each of the two poisoning approaches?
3. Other poising methods you can think of that might be more effective? Try and implement them and test it!

# Whitebox attack / Surrogate transferability

It has been noticed early on in the field that adversarial attacks actually transfer across neural networks. For instance, the research paper `Why Do Adversarial Attacks Transfer` by Ambra Demontis et. al. 

The idea is that we could craft an adversarial attack on a local and smaller **surrogate model** and still successfully carry out the same attack on a larger complex **target model**.

While the target model might be black-box, we can use our surrogate model as a white-box for gradient descent.


In [ ]:
data = ld_mnist()
print("Data loaded.")

## Surrogate CNN

Let's train our attack on the `BasicCNN`.

In [ ]:
# If data.train is a DataLoader, get its underlying dataset
train_dataset = data.train.dataset if hasattr(data.train, "dataset") else data.train
test_loader = data.test  # assuming this is already a DataLoader


# (a) Only 20% of training data
subset_size = int(0.2 * len(train_dataset))
subset, _ = torch.utils.data.random_split(train_dataset, [subset_size, len(train_dataset)-subset_size])
surrogate_loader_small = torch.utils.data.DataLoader(subset, batch_size=128, shuffle=True)

surrogate_20_model = Basic_CNN()
surrogate_20_model = train_model(surrogate_20_model, surrogate_loader_small, test_loader, epochs=1)
print(f"Surrogate (20% data) acc: {evaluate_model(surrogate_20_model, test_loader)}%")

# (b) Stronger regularization
surrogate_regualized_model = Basic_CNN(p_drop=0.5)
surrogate_regualized_model = train_model(surrogate_regualized_model, data.train, test_loader, epochs=1)
print(f"Surrogate (regularized) acc: {evaluate_model(surrogate_regualized_model, test_loader)}%")


The test accuracies achieved from these models are the surrogate models accuracies on the clean test set. The real test comes in the next section where we use these surrogates to generate new attack data to fool the big or Target CNN.

## Transfer

Now we are ready to transfer the adversarial examples to the target model.

In [ ]:
def transfer_attack(surrogate, target, loader, eps=0.3):
    device = "cpu"
    surrogate.eval(); target.eval()
    total, fooled_sur, fooled_tgt = 0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        # adversarial crafted on surrogate
        x_adv = fast_gradient_method(surrogate, x, eps, np.inf)
        # predictions
        pred_sur = surrogate(x_adv).argmax(1)
        pred_tgt = target(x_adv).argmax(1)
        # fooling stats
        fooled_sur += (pred_sur != y).sum().item()
        fooled_tgt += (pred_tgt != y).sum().item()
        total += y.size(0)
    return fooled_sur/total, fooled_tgt/total

rate_sur, rate_tgt = transfer_attack(surrogate_20_model, target_cnn, data.test)
print(f"Transfer (20% surrogate) Surrogate fooling rate: {rate_sur*100:.2f}%,\n Target fooling rate: {rate_tgt*100:.2f}%")
print("---")
rate_sur, rate_tgt = transfer_attack(surrogate_regualized_model, target_cnn, data.test)
print(f"Transfer (regularized surrogate) Surrogate fooling rate: {rate_sur*100:.2f}%,\n Target fooling rate: {rate_tgt*100:.2f}%")

The results from here shows how well the surrogate models themselves where at classifing the examples generated and the Target models own classification compared.

Interesting insgihts to check.
- How does different levels of regualization compare?
- How much surrogate data is needed to have a reasonable model for generating adversarial examples?

# Exercises

<!-- todo -->

Exercise 1 – Evasion on FashionMNIST
Train a CNN on the FashionMNIST dataset (you may adjust its complexity). Use CleverHans to generate adversarial examples against this model and visualize both the clean and perturbed images to see how small the changes look.


Exercise 2 – Poisoning the Model
Apply poisoning techniques like label flip to the same CNN. Retrain the model with poisoned data and evaluate how the attack affects overall accuracy and targeted misclassification.


Exercise 3 – Transferability with a Surrogate
Train a second, surrogate CNN with a different architecture or complexity. Craft adversarial or poisoned examples on the surrogate and then test them against the original (primary) model to measure how well the attacks transfer across models.

### Template code to begin with FashionMNIST

In [ ]:
def ld_mnistFashion():
    train_transforms = torchvision.transforms.Compose([torchvision.transforms.ToTensor()])
    test_transforms = torchvision.transforms.Compose([torchvision.transforms.ToTensor()])

    # Load MNISTFashion dataset
    train_dataset = torchvision.datasets.FashionMNIST(root="/tmp/data", train = True, download = True, transform=train_transforms)
    test_dataset = torchvision.datasets.FashionMNIST(root="/tmp/data", train=False, download = True, transform=test_transforms)

    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)
    
    return EasyDict(train=train_loader, test=test_loader)


ld_mnistFashion()
dataFashion = ld_mnistFashion()
print("Data loaded.")

fashion_cnn = Basic_CNN()
train_model(fashion_cnn, dataFashion.train, dataFashion.test, epochs=1)